# 🚀 Credit Risk Scoring — Azure ML AutoML Training

**Datathon Project — Notebook 02: Modeling**

## 🎯 Tujuan Notebook
1. Connect ke Azure ML Workspace via Python SDK v2
2. Submit AutoML classification job dengan konfigurasi strategis
3. Monitor training progress real-time
4. Retrieve best model & semua metrics evaluation
5. Register best model untuk deployment

## 📋 Strategi Modeling
- **Task:** Binary Classification (default vs non-default)
- **Primary metric:** `AUC_weighted` — robust untuk 22% imbalanced data
- **Algorithms:** XGBoost, LightGBM, RandomForest, LogisticRegression
- **Validation:** 5-fold cross-validation
- **Timeout:** 30 menit (cukup untuk 10-15 trials)

## 🏗️ Output yang Akan Dihasilkan
- Best model (XGBoost atau LightGBM expected)
- Leaderboard semua model trials
- Feature importance scores
- Confusion matrix, ROC curve, PR curve
- Model registered di Azure ML Model Registry

## 1. Setup & Install Dependencies

In [ ]:
# Install Azure ML SDK v2 (uncomment jika belum)
# !pip install azure-ai-ml azure-identity mltable --upgrade

In [ ]:
from azure.ai.ml import MLClient, automl, Input
from azure.ai.ml.constants import AssetTypes
from azure.identity import DefaultAzureCredential, InteractiveBrowserCredential
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

print('✅ Libraries loaded')

## 2. Connect ke Azure ML Workspace

**⚠️ GANTI 3 nilai di bawah** dengan info workspace kamu (yang sudah dicatat dari Azure Portal).

In [ ]:
# ⚠️ GANTI dengan info workspace kamu
SUBSCRIPTION_ID = 'YOUR_SUBSCRIPTION_ID_HERE'  # contoh: '12345678-abcd-1234-5678-123456789abc'
RESOURCE_GROUP = 'rg-datathon-credit'           # ganti dengan resource group kamu
WORKSPACE_NAME = 'mlw-credit-risk-umkm'         # ganti dengan workspace name kamu

# Auth — try DefaultAzureCredential first (works in compute instance), fallback to interactive
try:
    credential = DefaultAzureCredential()
    credential.get_token('https://management.azure.com/.default')
    print('✅ Using DefaultAzureCredential')
except Exception:
    credential = InteractiveBrowserCredential()
    print('🔐 Using InteractiveBrowserCredential — browser akan terbuka untuk login')

# Connect to workspace
ml_client = MLClient(
    credential=credential,
    subscription_id=SUBSCRIPTION_ID,
    resource_group_name=RESOURCE_GROUP,
    workspace_name=WORKSPACE_NAME
)

ws = ml_client.workspaces.get(WORKSPACE_NAME)
print(f'\n✅ Connected ke workspace: {ws.name}')
print(f'   Location: {ws.location}')
print(f'   Resource group: {ws.resource_group}')

## 3. Verifikasi Compute Cluster & Data Asset

In [ ]:
# Cek compute cluster
COMPUTE_NAME = 'cpu-cluster-automl'

try:
    compute = ml_client.compute.get(COMPUTE_NAME)
    print(f'✅ Compute cluster ditemukan: {compute.name}')
    print(f'   Type: {compute.type}')
    print(f'   VM size: {compute.size}')
    print(f'   Min nodes: {compute.min_instances}')
    print(f'   Max nodes: {compute.max_instances}')
    print(f'   State: {compute.provisioning_state}')
except Exception as e:
    print(f'❌ Error: {e}')
    print('   Pastikan compute cluster sudah dibuat di Azure ML Studio!')

In [ ]:
# Cek data asset
DATA_ASSET_NAME = 'credit_risk_processed'
DATA_ASSET_VERSION = '1'

try:
    data_asset = ml_client.data.get(name=DATA_ASSET_NAME, version=DATA_ASSET_VERSION)
    print(f'✅ Data asset ditemukan: {data_asset.name} v{data_asset.version}')
    print(f'   Type: {data_asset.type}')
    print(f'   Path: {data_asset.path}')
except Exception as e:
    print(f'❌ Error: {e}')
    print('   Pastikan data asset sudah di-register dengan tipe TABULAR!')

## 4. Konfigurasi AutoML Job

Kita akan submit AutoML classification job dengan konfigurasi yang **strategically chosen** untuk imbalanced credit risk data.

In [ ]:
from azure.ai.ml import automl

# Variabel target (HARUS sesuai dengan nama kolom di dataset!)
TARGET_COLUMN = 'loan_status'
EXPERIMENT_NAME = 'credit-risk-umkm-automl'

# Input data dari data asset yang sudah register
training_data = Input(
    type=AssetTypes.MLTABLE,
    path=f'azureml:{DATA_ASSET_NAME}:{DATA_ASSET_VERSION}'
)

# Build classification job
classification_job = automl.classification(
    compute=COMPUTE_NAME,
    experiment_name=EXPERIMENT_NAME,
    training_data=training_data,
    target_column_name=TARGET_COLUMN,
    primary_metric='AUC_weighted',  # Best untuk imbalanced data
    n_cross_validations=5,
    enable_model_explainability=True,  # PENTING untuk Responsible AI
)

# Limits — kontrol durasi & biaya
classification_job.set_limits(
    timeout_minutes=30,           # Max total durasi experiment
    trial_timeout_minutes=10,     # Max durasi per trial
    max_trials=20,                # Max jumlah algoritma yang dicoba
    max_concurrent_trials=2,      # Paralel 2 trial (sesuai max nodes cluster)
    enable_early_termination=True # Stop kalau improvement marginal
)

# Training settings — algoritma yang dicoba
classification_job.set_training(
    blocked_training_algorithms=['TabnetClassifier'],  # Skip yang slow
    enable_onnx_compatible_models=False,
    enable_stack_ensemble=True,    # Combine top models — sering pemenang
    enable_vote_ensemble=True
)

# Featurization — biarkan Azure auto-handle
classification_job.set_featurization(
    mode='auto'  # Auto encoding, scaling, missing handling
)

print('✅ AutoML job configuration ready')
print(f'   Experiment: {EXPERIMENT_NAME}')
print(f'   Target: {TARGET_COLUMN}')
print(f'   Primary metric: AUC_weighted')
print(f'   Compute: {COMPUTE_NAME}')
print(f'   Max trials: 20, Timeout: 30 min')

## 5. Submit AutoML Job

**⏱️ Estimasi durasi: 25-35 menit**

Setelah submit, kamu bisa:
- Monitor progress di cell selanjutnya
- Atau buka di Azure ML Studio → menu **Jobs** → klik experiment-mu untuk live UI

In [ ]:
# Submit job
returned_job = ml_client.jobs.create_or_update(classification_job)

print('🚀 AutoML job submitted!')
print(f'   Job name: {returned_job.name}')
print(f'   Status: {returned_job.status}')
print(f'\n📊 Monitor di Azure ML Studio:')
print(f'   {returned_job.studio_url}')

## 6. Monitor Progress

Cell ini akan **stream** log dan tunggu sampai job selesai. Kamu bisa biarkan running, atau **buka link Studio** di atas untuk live UI yang lebih nyaman.

In [ ]:
# OPSI A: Stream log (akan block sampai selesai, ~30 menit)
ml_client.jobs.stream(returned_job.name)

# OPSI B (alternatif): Cek status periodically tanpa blocking
# import time
# while True:
#     job = ml_client.jobs.get(returned_job.name)
#     print(f'Status: {job.status}')
#     if job.status in ['Completed', 'Failed', 'Canceled']:
#         break
#     time.sleep(60)

## 7. Retrieve Best Model & Metrics

Setelah job complete, kita ambil best model dan analisis hasilnya.

In [ ]:
# Refresh job info
completed_job = ml_client.jobs.get(returned_job.name)
print(f'Job status: {completed_job.status}')
print(f'Job name: {completed_job.name}')

if completed_job.status != 'Completed':
    print('⚠️ Job belum complete. Tunggu sebentar lalu re-run cell ini.')
else:
    print('✅ Job complete! Lanjut ke analisis hasil.')

In [ ]:
# Get best child run (algoritma pemenang)
from mlflow.tracking import MlflowClient
import mlflow

# Setup MLflow tracking ke Azure ML
MLFLOW_TRACKING_URI = ml_client.workspaces.get(WORKSPACE_NAME).mlflow_tracking_uri
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow_client = MlflowClient()

# Get parent job & best child
parent_run = mlflow_client.get_run(returned_job.name)
best_child_run_id = parent_run.data.tags.get('automl_best_child_run_id', None)

if best_child_run_id:
    best_run = mlflow_client.get_run(best_child_run_id)
    
    print('🏆 BEST MODEL TRIAL')
    print('=' * 60)
    print(f'Run ID: {best_child_run_id}')
    print(f'Algorithm: {best_run.data.tags.get("model_explanation", "N/A")}')
    print(f'\n📊 METRICS:')
    
    important_metrics = ['AUC_weighted', 'accuracy', 'f1_score_weighted', 
                          'precision_score_weighted', 'recall_score_weighted',
                          'log_loss', 'matthews_correlation']
    
    for metric in important_metrics:
        if metric in best_run.data.metrics:
            value = best_run.data.metrics[metric]
            print(f'   {metric:30s}: {value:.4f}')
else:
    print('⚠️ Best child run belum tersedia. Cek manual di Azure ML Studio.')

## 8. Leaderboard — Compare Semua Trials

In [ ]:
# Get semua child runs untuk leaderboard
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
if experiment:
    runs_df = mlflow.search_runs(
        experiment_ids=[experiment.experiment_id],
        filter_string=f"tags.mlflow.parentRunId = '{returned_job.name}'",
        order_by=['metrics.AUC_weighted DESC']
    )
    
    if not runs_df.empty:
        # Display top 10 leaderboard
        leaderboard_cols = ['tags.mlflow.runName', 'metrics.AUC_weighted', 
                            'metrics.accuracy', 'metrics.f1_score_weighted',
                            'metrics.precision_score_weighted', 'metrics.recall_score_weighted']
        
        available_cols = [c for c in leaderboard_cols if c in runs_df.columns]
        leaderboard = runs_df[available_cols].head(10).copy()
        leaderboard.columns = [c.replace('metrics.', '').replace('tags.mlflow.runName', 'algorithm') 
                                for c in leaderboard.columns]
        
        print('🏆 TOP 10 LEADERBOARD')
        print('=' * 100)
        print(leaderboard.to_string(index=False))
    else:
        print('⚠️ Tidak ada child runs ditemukan')
else:
    print('⚠️ Experiment belum ditemukan')

## 9. Download Best Model

In [ ]:
import os

# Download model artifact dari best run
OUTPUT_DIR = './best_model_artifacts'
os.makedirs(OUTPUT_DIR, exist_ok=True)

if best_child_run_id:
    mlflow_client.download_artifacts(
        run_id=best_child_run_id,
        path='outputs',
        dst_path=OUTPUT_DIR
    )
    print(f'✅ Best model artifacts downloaded ke: {OUTPUT_DIR}')
    print('\n📁 Files:')
    for root, dirs, files in os.walk(OUTPUT_DIR):
        for f in files:
            filepath = os.path.join(root, f)
            size_kb = os.path.getsize(filepath) / 1024
            print(f'   {filepath} ({size_kb:.1f} KB)')

## 10. Register Best Model untuk Deployment

**Critical step untuk skor Azure!** Model yang ter-register di Azure ML Model Registry bisa di-deploy ke endpoint, ditrack versioning-nya, dan terlihat di Studio.

In [ ]:
from azure.ai.ml.entities import Model
from azure.ai.ml.constants import AssetTypes

# Register model dari best run
MODEL_NAME = 'credit-risk-umkm-best-model'

model = Model(
    name=MODEL_NAME,
    path=f'azureml://jobs/{best_child_run_id}/outputs/artifacts/outputs/mlflow-model/',
    description='Best AutoML model for UMKM credit risk scoring — Datathon Project',
    type=AssetTypes.MLFLOW_MODEL,
    tags={
        'project': 'datathon-credit-risk',
        'task': 'binary-classification',
        'domain': 'financial-inclusion',
        'auto_ml': 'true'
    }
)

registered_model = ml_client.models.create_or_update(model)
print(f'✅ Model registered!')
print(f'   Name: {registered_model.name}')
print(f'   Version: {registered_model.version}')
print(f'   ID: {registered_model.id}')

## 11. Insight Strategis dari Hasil Training

**[ISI MANUAL berdasarkan hasil kamu]**

### Hasil Modeling:
- **Best algorithm:** [misal: VotingEnsemble / XGBoost / LightGBM]
- **AUC_weighted:** [angka — target > 0.85]
- **F1-Score:** [angka]
- **Total trials:** [berapa algoritma dicoba]

### Interpretasi:
1. **AUC > 0.85** → model **excellent**, bisa dipakai produksi
2. **AUC 0.75-0.85** → model **good**, masih bisa ditingkatkan
3. **AUC < 0.75** → perlu re-engineering features

### Bridge ke Notebook 03:
Setelah model registered, kita lanjut ke:
- **Responsible AI Dashboard** untuk fairness analysis & SHAP
- **Counterfactual analysis** untuk actionable recommendations
- **Azure OpenAI integration** untuk auto-generate insight narratives